## 07 Parks & Compounds

**笔记本**：`07 parks_compounds20260403.1.ipynb`

**库**：`geopandas`、`pandas`、`numpy`、`osmium`（pyosmium）、`matplotlib`、`shapely` 等。

**输入**：
- `../04 Transport/data_raw/guangdong.osm.pbf`、`shenzhen_boundary.geojson`
- `../04 Transport/data_out/sz_drive_edges.gpkg`、`../06 Buildings/data_out/sz_buildings.gpkg`（形态/邻近分析）
- `../00/11-public-facilities/SHP/…`（小区面）
- `../00/02-poi-aoi/2-AOI/…`（AOI、地块）
- `../00/15-residential-compound-data/…`（房地产小区面）

**做了什么 / 算了什么**：
（“哪些地方本身就是一个有边界、有内部组织、进入不一定方便的复合空间单元？”）
从 OSM PBF 提取「compound」面： 用 CompoundHandler 的 area() 回调，按标签规则得到 park / campus / commercial / industrial / residential（leisure、amenity、landuse 等组合）。
裁剪到深圳，再用外部图层补充 OSM 不足（ 叠加 00 的「小区面」「房地产小区」→ 一律标为 residential；叠加 AOI 中 fclass 映射到 park/commercial/industrial。）
把 OSM 候选面 + 外部更专门的数据源合并起来，形成更完整的 compound inventory。
去重： 同类型 + 质心约到 4 位小数，只保留 面积最大 的多边形。
形态与通达指标（投影 EPSG:32650）：
area_m2、perimeter_m
紧凑度 compactness = 4π·A/P²（限制在 [0,1]）
一个形状越接近圆形或规则紧凑形，compactness 越高；
越细长、破碎、边界曲折，compactness 越低。
去掉 面积 < 500 m² 的面
围合度 enclosure_ratio： compound 边界线与 06 建筑 union 相交长度 / 边界总长
如果一个 compound 四周很多地方都被建筑“围住”，那它的 enclosure_ratio 就高。
如果它边界大部分都是开敞的、面向道路或空地， 那这个值就低。
内部路网密度 internal_road_density： compound 内 04 道路边 总长度 / 面积
这个 compound 内部是否“有路可走、内部通达性如何”。
值高：内部道路较多，内部交通组织较发达
值低：内部可能更封闭、更少车行路、进入后可通行网络弱
在不同类型上含义可能不同
对工业园：内部路网高，说明货运/车行组织较强
对公园：内部如果只统计车行路，可能低，不代表不好，只代表“不是车行导向”
对住宅小区：内部车行/支路越少，地面配送可能更依赖入口后步行/短驳 
入口相关： 边界与路网求交得 n_entrances，entrance_scarcity = 1/(1+n_entrances)
公园活动 proxy： 对 park 为 log1p(area_m2) × compactness，其余类型为 0
哪些公园更可能是重要活动节点；哪些只是零碎绿地

**写出文件**：
- `data_out/sz_compounds.gpkg`

**典型输出信息**：OSM 提取数量、各数据源追加条数、`compound_type` 汇总、最终行数与保存路径。
定义高摩擦空间类型；构建场景类型 cards；解释为什么某些地方无人机更有补位价值；给选址模拟提供语义约束，例如新增候选 site 时，可以优先考虑：接近高 entrance_scarcity compound 的边缘；接近大型 park / campus / residential compound；接近 enclosure 高、内部进入难的地方


In [1]:
# ============================================================
#  07 Parks / Campuses / Commercial / Industrial Compounds
#  从 PBF 提取四类封闭/半封闭区域 polygon
#  计算形态指标 → 供 task typology 使用
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import osmium
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon, LineString
from shapely.prepared import prep as shapely_prep
from shapely.ops import unary_union
import warnings
warnings.filterwarnings("ignore")

OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

PBF = Path("../04 Transport/data_raw/guangdong.osm.pbf")
BOUNDARY = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")
EDGES_04 = Path("../04 Transport/data_out/sz_drive_edges.gpkg")
BUILDINGS_06 = Path("../06 Buildings/data_out/sz_buildings.gpkg")

# 00 原始数据
XIAOQU_SHP = Path("../00/11-public-facilities/SHP/shenzhen_residential_compound_polygons.shp")
AOI_SHP = Path("../00/02-poi-aoi/2-AOI/shenzhen_aoi.shp")
LAND_SHP = Path("../00/02-poi-aoi/2-AOI/shenzhen_land_parcel_cadastre.shp")
FANGDICHAN_SHP = Path("../00/15-residential-compound-data/property-polygons-baidu/shenzhen_real_estate_residential_compounds.shp")

shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union
bbox = shenzhen_geom.bounds

print(f"PBF: {PBF.exists()}")
print(f"04 edges: {EDGES_04.exists()}")
print(f"06 buildings: {BUILDINGS_06.exists()}")
print(f"小区面: {XIAOQU_SHP.exists()} | AOI: {AOI_SHP.exists()} | 地块: {LAND_SHP.exists()}")
print(f"房地产小区: {FANGDICHAN_SHP.exists()}")

PBF: True
04 edges: True
06 buildings: True
小区面: True | AOI: True | 地块: True
房地产小区: True


In [2]:
# ============================================================
#  1. 从 PBF 提取四类 compound polygon
#  用 pyosmium area() 回调解析 multipolygon relations + closed ways
# ============================================================

# 分类规则: OSM tag → compound_type
TAG_RULES = [
    # (tag_key, tag_values, compound_type)
    ("leisure",  {"park", "nature_reserve", "recreation_ground"}, "park"),
    ("leisure",  {"garden"},                                      "park"),
    ("amenity",  {"school", "university", "college", "kindergarten"}, "campus"),
    ("landuse",  {"commercial", "retail"},                        "commercial"),
    ("amenity",  {"marketplace"},                                 "commercial"),
    ("landuse",  {"industrial"},                                  "industrial"),
    ("landuse",  {"residential"},                                 "residential"),
]

class CompoundHandler(osmium.SimpleHandler):
    def __init__(self, minx, miny, maxx, maxy):
        super().__init__()
        self.minx, self.miny, self.maxx, self.maxy = minx, miny, maxx, maxy
        self.features = []

    def _classify(self, tags):
        for key, values, ctype in TAG_RULES:
            v = tags.get(key)
            if v and v in values:
                return ctype
        return None

    def area(self, a):
        tags = dict(a.tags)
        ctype = self._classify(tags)
        if ctype is None:
            return
        try:
            outer = a.outer_rings()
        except Exception:
            return
        for ring in outer:
            coords = [(n.lon, n.lat) for n in ring if n.location.valid()]
            if len(coords) < 4:
                continue
            lons = [c[0] for c in coords]
            lats = [c[1] for c in coords]
            if max(lons) < self.minx or min(lons) > self.maxx:
                continue
            if max(lats) < self.miny or min(lats) > self.maxy:
                continue
            try:
                poly = Polygon(coords)
                if not poly.is_valid:
                    poly = poly.buffer(0)
                if poly.is_empty or poly.area <= 0:
                    continue
                self.features.append({
                    "compound_type": ctype,
                    "name": tags.get("name", ""),
                    "osm_leisure": tags.get("leisure", ""),
                    "osm_amenity": tags.get("amenity", ""),
                    "osm_landuse": tags.get("landuse", ""),
                    "geometry": poly,
                })
            except Exception:
                pass

print("Extracting compounds from PBF ...")
ch = CompoundHandler(*bbox)
ch.apply_file(str(PBF), locations=True)
print(f"  bbox-filtered: {len(ch.features)} polygons")

if ch.features:
    compounds = gpd.GeoDataFrame(ch.features, crs=4326)
    compounds = gpd.clip(compounds, shenzhen_geom)
else:
    compounds = gpd.GeoDataFrame()

print(f"  clipped (OSM): {len(compounds)} compounds")

# ── 补充: 00 小区面 → residential ──
if XIAOQU_SHP.exists():
    xq = gpd.read_file(XIAOQU_SHP).to_crs(4326)
    xq = gpd.clip(xq, shenzhen_geom)
    xq_compounds = gpd.GeoDataFrame({
        "compound_type": "residential",
        "name": xq["name"].fillna(""),
        "osm_leisure": "", "osm_amenity": "", "osm_landuse": "residential",
        "geometry": xq.geometry,
    }, crs=4326)
    print(f"  小区面: {len(xq_compounds)} residential polygons added")
    compounds = pd.concat([compounds, xq_compounds], ignore_index=True)
    del xq, xq_compounds

# ── 补充: 00 房地产小区面 (9,118 polygon, 带房价) ──
if FANGDICHAN_SHP.exists():
    fdc = gpd.read_file(FANGDICHAN_SHP).to_crs(4326)
    fdc = gpd.clip(fdc, shenzhen_geom)
    fdc_compounds = gpd.GeoDataFrame({
        "compound_type": "residential",
        "name": fdc["name"].fillna(""),
        "osm_leisure": "", "osm_amenity": "", "osm_landuse": "residential",
        "geometry": fdc.geometry,
    }, crs=4326)
    print(f"  房地产小区: {len(fdc_compounds)} residential polygons added")
    compounds = pd.concat([compounds, fdc_compounds], ignore_index=True)
    del fdc, fdc_compounds

# ── 补充: 00 AOI → 按 fclass 补充缺失的类别 ──
if AOI_SHP.exists():
    aoi = gpd.read_file(AOI_SHP).to_crs(4326)
    aoi = gpd.clip(aoi, shenzhen_geom)

    AOI_MAP = {
        "park": "park", "recreation_ground": "park", "nature_reserve": "park",
        "commercial": "commercial", "retail": "commercial",
        "industrial": "industrial",
    }
    aoi["compound_type"] = aoi["fclass"].map(AOI_MAP)
    aoi_valid = aoi[aoi["compound_type"].notna()].copy()
    aoi_compounds = gpd.GeoDataFrame({
        "compound_type": aoi_valid["compound_type"],
        "name": aoi_valid["name"].fillna(""),
        "osm_leisure": "", "osm_amenity": "", "osm_landuse": aoi_valid["fclass"],
        "geometry": aoi_valid.geometry,
    }, crs=4326)
    print(f"  AOI: {len(aoi_compounds)} compound polygons added (park/commercial/industrial)")
    compounds = pd.concat([compounds, aoi_compounds], ignore_index=True)
    del aoi, aoi_valid, aoi_compounds

# ── 去重: 相同位置 + 相同类型的 polygon 只保留面积最大的 ──
compounds["_cx"] = compounds.geometry.centroid.x.round(4)
compounds["_cy"] = compounds.geometry.centroid.y.round(4)
compounds["_area"] = compounds.geometry.area
compounds = compounds.sort_values("_area", ascending=False).drop_duplicates(
    subset=["compound_type", "_cx", "_cy"], keep="first"
).drop(columns=["_cx", "_cy", "_area"]).reset_index(drop=True)

print(f"\n  Final (after merge + dedup): {len(compounds)}")
print(compounds["compound_type"].value_counts().to_string())
del ch

Extracting compounds from PBF ...
  bbox-filtered: 20016 polygons
  clipped (OSM): 12151 compounds
  小区面: 3612 residential polygons added
  房地产小区: 9118 residential polygons added
  AOI: 4931 compound polygons added (park/commercial/industrial)

  Final (after merge + dedup): 20709
compound_type
residential    13624
industrial      2659
commercial      1559
campus          1539
park            1328


In [3]:
# ============================================================
#  2. 形态指标计算
#  area / perimeter / compactness / enclosure_ratio / internal_road_density
# ============================================================

# 投影到 UTM 50N 计算面积和周长
compounds_proj = compounds.to_crs(32650)

compounds["area_m2"] = compounds_proj.geometry.area
compounds["perimeter_m"] = compounds_proj.geometry.length

# 紧凑度: 4π × area / perimeter² (圆=1, 越小越不规则)
compounds["compactness"] = (
    4 * np.pi * compounds["area_m2"] / (compounds["perimeter_m"] ** 2)
).clip(0, 1)

# 过滤掉过小的 polygon (< 500 m², 可能是标注噪声)
before = len(compounds)
compounds = compounds[compounds["area_m2"] >= 500].reset_index(drop=True)
print(f"Filtered: {before} -> {len(compounds)} (removed area < 500 m²)")

# ── 围合度: compound 边界上被建筑覆盖的比例 ──
print("\nCalculating enclosure ratio (building overlap on boundary) ...")

if BUILDINGS_06.exists():
    bldg = gpd.read_file(BUILDINGS_06)
    bldg_union = unary_union(bldg.geometry)
    bldg_prep = shapely_prep(bldg_union)

    enclosure_ratios = []
    for idx, row in compounds.iterrows():
        boundary_line = row.geometry.boundary
        try:
            if bldg_prep.intersects(boundary_line):
                overlap = boundary_line.intersection(bldg_union)
                ratio = overlap.length / boundary_line.length if boundary_line.length > 0 else 0
            else:
                ratio = 0
        except Exception:
            ratio = 0
        enclosure_ratios.append(min(ratio, 1.0))
    compounds["enclosure_ratio"] = enclosure_ratios
    del bldg, bldg_union, bldg_prep
    print(f"  Done. Mean enclosure: {compounds['enclosure_ratio'].mean():.3f}")
else:
    compounds["enclosure_ratio"] = np.nan
    print("  Skipped (06 buildings not found)")

# ── 内部路网密度: compound 内道路总长 / 面积 ──
print("\nCalculating internal road density ...")

if EDGES_04.exists():
    edges = gpd.read_file(EDGES_04)
    edges_proj = edges.to_crs(32650)

    road_densities = []
    compounds_proj2 = compounds.to_crs(32650)
    for idx, row in compounds_proj2.iterrows():
        poly = row.geometry
        try:
            clipped = gpd.clip(edges_proj, poly)
            total_len = clipped.geometry.length.sum()
            density = total_len / row.geometry.area if row.geometry.area > 0 else 0
        except Exception:
            density = 0
        road_densities.append(density)
    compounds["internal_road_density"] = road_densities

    # ── 入口稀缺 proxy: compound 边界与路网的交点数 ──
    print("\nCalculating entrance scarcity ...")
    edges_for_entrance = gpd.read_file(EDGES_04)
    edges_union = unary_union(edges_for_entrance.geometry)
    del edges_for_entrance
    entrance_counts = []
    for idx, row in compounds.iterrows():
        boundary_line = row.geometry.boundary
        try:
            inter = boundary_line.intersection(edges_union)
            if inter.is_empty:
                n = 0
            elif inter.geom_type == "Point":
                n = 1
            elif inter.geom_type == "MultiPoint":
                n = len(inter.geoms)
            else:
                n = len(list(inter.geoms)) if hasattr(inter, "geoms") else 1
        except Exception:
            n = 0
        entrance_counts.append(n)
    compounds["n_entrances"] = entrance_counts
    compounds["entrance_scarcity"] = 1 / (1 + compounds["n_entrances"])
    del edges_union
    print(f"  Done. Mean entrances: {compounds['n_entrances'].mean():.1f}, mean scarcity: {compounds['entrance_scarcity'].mean():.3f}")

    del edges_proj
    print(f"\n  Done. Mean road density: {compounds['internal_road_density'].mean():.4f} m/m²")
else:
    compounds["internal_road_density"] = np.nan
    compounds["n_entrances"] = np.nan
    compounds["entrance_scarcity"] = np.nan
    print("  Skipped (04 edges not found)")

# ── 公园活动 proxy: 面积 × 紧凑度 (越大越规则的公园, 游客容量越大) ──
compounds["park_activity_proxy"] = np.where(
    compounds["compound_type"] == "park",
    np.log1p(compounds["area_m2"]) * compounds["compactness"],
    0,
)
print(f"\nParks with activity proxy > 0: {(compounds['park_activity_proxy'] > 0).sum()}")

print(f"\nFinal compounds: {len(compounds)}")

Filtered: 20709 -> 19637 (removed area < 500 m²)

Calculating enclosure ratio (building overlap on boundary) ...
  Done. Mean enclosure: 0.203

Calculating internal road density ...

Calculating entrance scarcity ...
  Done. Mean entrances: 1.3, mean scarcity: 0.752

  Done. Mean road density: 0.0088 m/m²

Parks with activity proxy > 0: 1172

Final compounds: 19637


In [4]:
# ============================================================
#  3. 保存输出 + 统计摘要
# ============================================================

compounds.to_file(OUT / "sz_compounds.gpkg", driver="GPKG")
print(f"Compounds -> data_out/sz_compounds.gpkg  ({len(compounds):,} rows)")

print("\n=== Summary by Type ===")
summary = compounds.groupby("compound_type").agg(
    count=("area_m2", "count"),
    total_area_km2=("area_m2", lambda x: x.sum() / 1e6),
    avg_area_m2=("area_m2", "mean"),
    avg_compactness=("compactness", "mean"),
    avg_enclosure=("enclosure_ratio", "mean"),
    avg_road_density=("internal_road_density", "mean"),
).round(3)
print(summary.to_string())

print("\n=== Top 10 Largest Compounds ===")
top10 = compounds.nlargest(10, "area_m2")[["name", "compound_type", "area_m2", "compactness", "enclosure_ratio"]]
top10["area_ha"] = (top10["area_m2"] / 10000).round(1)
print(top10[["name", "compound_type", "area_ha", "compactness", "enclosure_ratio"]].to_string())

print("\n=== High-Enclosure Compounds (>0.3) ===")
high_enc = compounds[compounds["enclosure_ratio"] > 0.3]
print(f"  {len(high_enc)} compounds with enclosure > 0.3")
print(high_enc["compound_type"].value_counts().to_string())

Compounds -> data_out/sz_compounds.gpkg  (19,637 rows)

=== Summary by Type ===
               count  total_area_km2  avg_area_m2  avg_compactness  avg_enclosure  avg_road_density
compound_type                                                                                      
campus          1538          37.727    24530.111            0.693          0.158             0.003
commercial      1553          38.204    24599.998            0.686          0.154             0.012
industrial      2656         140.255    52806.708            0.676          0.203             0.006
park            1172         153.334   130831.336            0.528          0.059             0.003
residential    12718         344.688    27102.388            0.658          0.228             0.010

=== Top 10 Largest Compounds ===
        name compound_type  area_ha  compactness  enclosure_ratio
0    凤凰山森林公园          park   2437.2     0.100695         0.010886
1    凤凰山森林公园          park   2344.2     0.062259      